### Imports Libraries

In [1]:
import os
import shutil
from pathlib import Path

import pandas as pd
import numpy as np

from PIL import Image

### Dataset Paths

In [7]:
DATASET01_PATH = Path("../datasets/dataset_01")
DATASET02_PATH = Path("../datasets/dataset_02/Original")
DATASET03_PATH = Path("../datasets/dataset_03")

PROCESSED_PATH = Path("../processed")

### Standardize Class Names

In [8]:
CLASS_MAPPING = {
    "Healthy Rice Leaf": "healthy",

    "Bacterial Leaf Blight": "bacterial_leaf_blight",
    "Bacterial Blight Disease": "bacterial_leaf_blight",

    "Brown Spot": "brown_spot",
    "Brown Spot Disease": "brown_spot",

    "Leaf Blast": "blast",
    "Blast Disease": "blast",

    "Leaf scald": "leaf_scald",

    "Sheath Blight": "sheath_blight",

    "Narrow Brown Leaf Spot": "narrow_brown_leaf_spot",

    "Rice Hispa": "rice_hispa",

    "False Smut Disease": "false_smut"
}

### Create Processed Class Folders

In [9]:
for class_name in set(CLASS_MAPPING.values()):

    (PROCESSED_PATH / class_name).mkdir(
        parents=True,
        exist_ok=True
    )

### Merge Datasets

In [10]:
def copy_images(dataset_path):

    for folder in dataset_path.iterdir():

        if not folder.is_dir():
            continue

        original_class = folder.name

        if original_class not in CLASS_MAPPING:
            continue

        target_class = CLASS_MAPPING[original_class]

        target_dir = PROCESSED_PATH / target_class

        for image_file in folder.glob("*"):

            if image_file.is_file():

                new_name = (
                    f"{dataset_path.name}_"
                    f"{image_file.name}"
                )

                shutil.copy(
                    image_file,
                    target_dir / new_name
                )

In [11]:
copy_images(DATASET01_PATH)
copy_images(DATASET02_PATH)
copy_images(DATASET03_PATH)

### Verify Counts

In [12]:
processed_counts = {}

for class_folder in PROCESSED_PATH.iterdir():

    if class_folder.is_dir():

        processed_counts[class_folder.name] = len(
            list(class_folder.glob("*"))
        )

pd.DataFrame(
    processed_counts.items(),
    columns=["Class","Images"]
)

,Class,Images
0,bacterial_leaf_blight,866
1,blast,989
2,brown_spot,963
3,false_smut,50
4,healthy,810
5,leaf_scald,817
6,narrow_brown_leaf_spot,117
7,rice_hispa,215
8,sheath_blight,903
9,test,0


### Remove Corrupted Images

In [13]:
from PIL import Image

corrupted_images = []

for class_folder in PROCESSED_PATH.iterdir():

    if not class_folder.is_dir():
        continue

    if class_folder.name in ["train", "valid", "test"]:
        continue

    for image_file in class_folder.glob("*"):

        try:
            img = Image.open(image_file)
            img.verify()

        except Exception:

            corrupted_images.append(image_file)

In [14]:
print(f"Corrupted Images Found: {len(corrupted_images)}")

Corrupted Images Found: 0


### Resize Images to 224×224

In [15]:
IMAGE_SIZE = (224, 224)

In [16]:
from PIL import Image

for class_folder in PROCESSED_PATH.iterdir():

    if not class_folder.is_dir():
        continue

    if class_folder.name in ["train", "valid", "test"]:
        continue

    for image_file in class_folder.glob("*"):

        try:

            image = Image.open(image_file)

            image = image.convert("RGB")

            image = image.resize(
                IMAGE_SIZE,
                Image.Resampling.LANCZOS
            )

            image.save(image_file)

        except Exception as e:

            print(
                f"Error processing {image_file.name}: {e}"
            )

In [17]:
sample_image = next(
    (PROCESSED_PATH / "healthy").glob("*")
)

img = Image.open(sample_image)

print(img.size)

(224, 224)


Train / Validation / Test Split

In [18]:
from sklearn.model_selection import train_test_split
import shutil

In [19]:
TRAIN_PATH = PROCESSED_PATH / "train"
VALID_PATH = PROCESSED_PATH / "valid"
TEST_PATH = PROCESSED_PATH / "test"

TRAIN_PATH.mkdir(exist_ok=True)
VALID_PATH.mkdir(exist_ok=True)
TEST_PATH.mkdir(exist_ok=True)

In [20]:
CLASS_NAMES = [
    folder.name
    for folder in PROCESSED_PATH.iterdir()
    if folder.is_dir()
    and folder.name not in ["train", "valid", "test"]
]

CLASS_NAMES

['bacterial_leaf_blight',
 'blast',
 'brown_spot',
 'false_smut',
 'healthy',
 'leaf_scald',
 'narrow_brown_leaf_spot',
 'rice_hispa',
 'sheath_blight']

In [21]:
for class_name in CLASS_NAMES:

    (TRAIN_PATH / class_name).mkdir(
        parents=True,
        exist_ok=True
    )

    (VALID_PATH / class_name).mkdir(
        parents=True,
        exist_ok=True
    )

    (TEST_PATH / class_name).mkdir(
        parents=True,
        exist_ok=True
    )

In [22]:
for class_name in CLASS_NAMES:

    class_dir = PROCESSED_PATH / class_name

    images = list(class_dir.glob("*"))

    train_imgs, temp_imgs = train_test_split(
        images,
        test_size=0.30,
        random_state=42
    )

    valid_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=0.50,
        random_state=42
    )

    for img in train_imgs:

        shutil.copy(
            img,
            TRAIN_PATH / class_name / img.name
        )

    for img in valid_imgs:

        shutil.copy(
            img,
            VALID_PATH / class_name / img.name
        )

    for img in test_imgs:

        shutil.copy(
            img,
            TEST_PATH / class_name / img.name
        )

### Verify Split

In [23]:
def count_images(folder_path):

    counts = {}

    for class_folder in folder_path.iterdir():

        if class_folder.is_dir():

            counts[class_folder.name] = len(
                list(class_folder.glob("*"))
            )

    return counts

In [24]:
train_counts = count_images(TRAIN_PATH)
valid_counts = count_images(VALID_PATH)
test_counts = count_images(TEST_PATH)

pd.DataFrame({
    "Train": train_counts,
    "Validation": valid_counts,
    "Test": test_counts
})

,Train,Validation,Test
bacterial_leaf_blight,606,130,130
blast,692,148,149
brown_spot,674,144,145
false_smut,35,7,8
healthy,567,121,122
leaf_scald,571,123,123
narrow_brown_leaf_spot,81,18,18
rice_hispa,150,32,33
sheath_blight,632,135,136
